In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Load all datasets
df_orders = spark.read.option("header", True).option("inferSchema", True).csv("Files/Raw/olist_orders_dataset.csv")
df_items = spark.read.option("header", True).option("inferSchema", True).csv("Files/Raw/olist_order_items_dataset.csv")
df_customers = spark.read.option("header", True).option("inferSchema", True).csv("Files/Raw/olist_customers_dataset.csv")
df_products = spark.read.option("header", True).option("inferSchema", True).csv("Files/Raw/olist_products_dataset.csv")
df_sellers = spark.read.option("header", True).option("inferSchema", True).csv("Files/Raw/olist_sellers_dataset.csv")
df_payments = spark.read.option("header", True).option("inferSchema", True).csv("Files/Raw/olist_order_payments_dataset.csv")
df_reviews = spark.read.option("header", True).option("inferSchema", True).csv("Files/Raw/olist_order_reviews_dataset.csv")
df_translation = spark.read.option("header", True).option("inferSchema", True).csv("Files/Raw/product_category_name_translation.csv")

print("Orders:", df_orders.count())
print("Items:", df_items.count())
print("Customers:", df_customers.count())
print("Products:", df_products.count())
print("Sellers:", df_sellers.count())
print("Payments:", df_payments.count())
print("Reviews:", df_reviews.count())
print("Translation:", df_translation.count())
print("All files loaded!")

StatementMeta(, 4f8585c1-33b7-414f-8f49-7e4d9d98730e, 3, Finished, Available, Finished, False)

Orders: 99441
Items: 112650
Customers: 99441
Products: 32951
Sellers: 3095
Payments: 103886
Reviews: 104162
Translation: 71
All files loaded!


In [2]:
from pyspark.sql.functions import col, round as spark_round, to_timestamp, month, year, datediff

# Join all tables together
df_master = df_orders \
    .join(df_items, on="order_id", how="inner") \
    .join(df_customers, on="customer_id", how="left") \
    .join(df_products, on="product_id", how="left") \
    .join(df_translation, on="product_category_name", how="left") \
    .join(df_sellers, on="seller_id", how="left")

# Clean and cast
df_master = df_master \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date")))

# Add derived columns
df_master = df_master \
    .withColumn("total_revenue", spark_round(col("price") + col("freight_value"), 2)) \
    .withColumn("order_year", year(col("order_purchase_timestamp"))) \
    .withColumn("order_month", month(col("order_purchase_timestamp"))) \
    .withColumn("delivery_days", datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp"))) \
    .withColumn("is_late", (
        col("order_delivered_customer_date") > col("order_estimated_delivery_date")
    ).cast("integer"))

# Filter cancelled
df_master = df_master.filter(col("order_status") != "canceled")

print(f"Master table rows: {df_master.count()}")
df_master.show(3)

StatementMeta(, 4f8585c1-33b7-414f-8f49-7e4d9d98730e, 4, Finished, Available, Finished, False)

Master table rows: 112108
+--------------------+---------------------+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-----+-------------+--------------------+------------------------+-------------+--------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----------------------------+----------------------+--------------+------------+-------------+----------+-----------+-------------+-------+
|           seller_id|product_category_name|          product_id|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|shipping_limit_date|price|freight_value|  cust

In [3]:
df_master.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("clean_orders_master")

print("Silver layer saved: clean_orders_master")

StatementMeta(, 4f8585c1-33b7-414f-8f49-7e4d9d98730e, 5, Finished, Available, Finished, False)

Silver layer saved: clean_orders_master


In [4]:
gold_category = spark.sql("""
    SELECT 
        COALESCE(product_category_name_english, 'uncategorized') AS category,
        COUNT(DISTINCT order_id)         AS total_orders,
        ROUND(SUM(total_revenue), 2)     AS total_revenue,
        ROUND(AVG(total_revenue), 2)     AS avg_order_value,
        ROUND(AVG(delivery_days), 1)     AS avg_delivery_days
    FROM clean_orders_master
    GROUP BY product_category_name_english
    ORDER BY total_revenue DESC
    LIMIT 20
""")

gold_category.show(10)
gold_category.write.mode("overwrite").format("delta").saveAsTable("gold_sales_by_category")
print("Gold 1 saved: gold_sales_by_category")

StatementMeta(, 4f8585c1-33b7-414f-8f49-7e4d9d98730e, 6, Finished, Available, Finished, False)

+--------------------+------------+-------------+---------------+-----------------+
|            category|total_orders|total_revenue|avg_order_value|avg_delivery_days|
+--------------------+------------+-------------+---------------+-----------------+
|       health_beauty|        8800|   1437665.78|         149.23|             11.9|
|       watches_gifts|        5604|   1298292.47|         217.47|             12.6|
|      bed_bath_table|        9399|   1240386.13|         111.78|             12.8|
|      sports_leisure|        7673|   1147244.63|         133.56|             12.1|
|computers_accesso...|        6654|   1050941.58|         135.07|             13.2|
|     furniture_decor|        6425|    899626.04|         108.41|             12.8|
|          housewares|        5847|    772035.14|         111.65|             10.9|
|          cool_stuff|        3617|    704176.47|         186.29|             12.3|
|                auto|        3873|     678652.6|         161.39|           

In [5]:
gold_state = spark.sql("""
    SELECT 
        customer_state                   AS state,
        COUNT(DISTINCT order_id)         AS total_orders,
        ROUND(SUM(total_revenue), 2)     AS total_revenue,
        ROUND(AVG(total_revenue), 2)     AS avg_order_value,
        ROUND(AVG(delivery_days), 1)     AS avg_delivery_days
    FROM clean_orders_master
    GROUP BY customer_state
    ORDER BY total_revenue DESC
""")

gold_state.show(10)
gold_state.write.mode("overwrite").format("delta").saveAsTable("gold_sales_by_state")
print("Gold 2 saved: gold_sales_by_state")

StatementMeta(, 4f8585c1-33b7-414f-8f49-7e4d9d98730e, 7, Finished, Available, Finished, False)

+-----+------------+-------------+---------------+-----------------+
|state|total_orders|total_revenue|avg_order_value|avg_delivery_days|
+-----+------------+-------------+---------------+-----------------+
|   SP|       41128|   5879496.03|         124.68|              8.7|
|   RJ|       12698|   2115981.14|         145.83|             15.1|
|   MG|       11497|   1843266.62|          141.0|             11.9|
|   RS|        5416|    877561.34|         141.27|             15.1|
|   PR|        4982|    794196.61|         138.87|             11.9|
|   SC|        3599|     608023.7|         146.12|             14.9|
|   BA|        3344|    606908.66|         160.35|             19.2|
|   DF|        2120|    351327.21|         146.57|             12.9|
|   GO|        1998|    340544.37|          146.6|             15.3|
|   ES|        2018|    323081.03|         143.72|             15.6|
+-----+------------+-------------+---------------+-----------------+
only showing top 10 rows

Gold 2 s

In [6]:
gold_monthly = spark.sql("""
    SELECT 
        order_year,
        order_month,
        COUNT(DISTINCT order_id)         AS total_orders,
        ROUND(SUM(total_revenue), 2)     AS total_revenue,
        ROUND(AVG(total_revenue), 2)     AS avg_order_value,
        SUM(is_late)                     AS late_deliveries
    FROM clean_orders_master
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
""")

gold_monthly.show(10)
gold_monthly.write.mode("overwrite").format("delta").saveAsTable("gold_sales_by_month")
print("Gold 3 saved: gold_sales_by_month")

StatementMeta(, 4f8585c1-33b7-414f-8f49-7e4d9d98730e, 8, Finished, Available, Finished, False)

+----------+-----------+------------+-------------+---------------+---------------+
|order_year|order_month|total_orders|total_revenue|avg_order_value|late_deliveries|
+----------+-----------+------------+-------------+---------------+---------------+
|      2016|          9|           2|       279.69|          55.94|              3|
|      2016|         10|         296|     53495.01|         153.28|              3|
|      2016|         12|           1|        19.62|          19.62|              0|
|      2017|          1|         787|    136943.46|          143.7|             26|
|      2017|          2|        1718|    283561.69|         146.47|             62|
|      2017|          3|        2617|    425617.96|         143.06|            163|
|      2017|          4|        2377|    405848.61|         152.57|            193|
|      2017|          5|        3640|    582710.83|         141.92|            151|
|      2017|          6|        3205|    499652.24|         139.92|         

In [7]:
# Load payments and join with orders
df_payments_clean = df_payments \
    .withColumn("payment_value", col("payment_value").cast("double"))

df_pay_master = df_orders \
    .join(df_payments_clean, on="order_id", how="inner") \
    .filter(col("order_status") != "canceled")

df_pay_master.createOrReplaceTempView("payments_master")

gold_payments = spark.sql("""
    SELECT 
        payment_type,
        COUNT(DISTINCT order_id)          AS total_orders,
        ROUND(SUM(payment_value), 2)      AS total_payment_value,
        ROUND(AVG(payment_value), 2)      AS avg_payment_value,
        ROUND(AVG(payment_installments), 1) AS avg_installments
    FROM payments_master
    GROUP BY payment_type
    ORDER BY total_payment_value DESC
""")

gold_payments.show()
gold_payments.write.mode("overwrite").format("delta").saveAsTable("gold_payments")
print("Gold 4 saved: gold_payments")

StatementMeta(, 4f8585c1-33b7-414f-8f49-7e4d9d98730e, 9, Finished, Available, Finished, False)

+------------+------------+-------------------+-----------------+----------------+
|payment_type|total_orders|total_payment_value|avg_payment_value|avg_installments|
+------------+------------+-------------------+-----------------+----------------+
| credit_card|       76061|      1.244470888E7|           162.99|             3.5|
|      boleto|       19689|         2851857.17|           144.85|             1.0|
|     voucher|        3772|          353771.95|             62.5|             1.0|
|  debit_card|        1521|          215278.52|           141.44|             1.0|
+------------+------------+-------------------+-----------------+----------------+

Gold 4 saved: gold_payments


In [8]:
df_reviews_clean = df_reviews \
    .withColumn("review_score", col("review_score").cast("integer"))

df_reviews_clean.createOrReplaceTempView("reviews_master")

gold_reviews = spark.sql("""
    SELECT 
        review_score,
        COUNT(*) AS total_reviews,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
    FROM reviews_master
    GROUP BY review_score
    ORDER BY review_score DESC
""")

gold_reviews.show()
gold_reviews.write.mode("overwrite").format("delta").saveAsTable("gold_reviews")
print("Gold 5 saved: gold_reviews")

StatementMeta(, 4f8585c1-33b7-414f-8f49-7e4d9d98730e, 10, Finished, Available, Finished, False)

+------------+-------------+------------+
|review_score|total_reviews|pct_of_total|
+------------+-------------+------------+
|          90|            1|        0.00|
|           5|        57328|       55.04|
|           4|        19142|       18.38|
|           3|         8179|        7.85|
|           2|         3151|        3.03|
|           1|        11424|       10.97|
|           0|            2|        0.00|
|        NULL|         4935|        4.74|
+------------+-------------+------------+

Gold 5 saved: gold_reviews
